# Thin the annotation pool

Each match folder in `MyDrive/Playbook/annotation_pool` holds 75 frames: 3 clips x 25 consecutive seconds.
This keeps **25 evenly spaced frames per match** (about every 3rd second, spread over all three clips),
copies them to `MyDrive/Playbook/annotation_pool_25/<match>/` with the original names, and makes one
zip per match for uploading to Roboflow. The original folders are not touched.

Run the single cell below (it asks for Drive access once). No GPU needed.

In [ ]:
# Thin the annotation pool: 25 evenly spaced frames per match (about every 3rd second)
import os, re, shutil, csv
import numpy as np
from google.colab import drive
drive.mount('/content/drive')

SRC = '/content/drive/MyDrive/Playbook/annotation_pool'
DST = '/content/drive/MyDrive/Playbook/annotation_pool_25'
PER_MATCH = 25          # frames kept per match
MAKE_ZIPS = True        # one zip per match, easy to upload to Roboflow from a tablet

if not os.path.isdir(SRC):  # fall back to searching for the folder
    hits = [r for r, d, f in os.walk('/content/drive/MyDrive') if r.endswith('/annotation_pool')]
    if not hits:
        raise SystemExit('annotation_pool folder not found in MyDrive')
    SRC = hits[0]; DST = SRC + '_25'
print('source:', SRC)

PAT = re.compile(r'clip(\d+)_(\d+)m(\d+)s_(\d+)', re.I)
def key(name):
    m = PAT.search(name)
    if not m:
        return (999, 0, name)
    clip, mm, ss, i = map(int, m.groups())
    return (clip, mm * 60 + ss, i)

os.makedirs(DST, exist_ok=True)
rows = []
for match in sorted(d for d in os.listdir(SRC) if os.path.isdir(f'{SRC}/{d}') and not d.startswith('_')):
    imgs = sorted((f for f in os.listdir(f'{SRC}/{match}') if f.lower().endswith(('.jpg', '.jpeg', '.png'))), key=key)
    if not imgs:
        continue
    n = min(PER_MATCH, len(imgs))
    pick = [imgs[i] for i in sorted(set(np.round(np.linspace(0, len(imgs) - 1, n)).astype(int)))]
    os.makedirs(f'{DST}/{match}', exist_ok=True)
    for f in pick:
        if not os.path.exists(f'{DST}/{match}/{f}'):
            shutil.copy2(f'{SRC}/{match}/{f}', f'{DST}/{match}/{f}')
        rows.append({'match': match, 'file': f})
    if MAKE_ZIPS:
        shutil.make_archive(f'{DST}/{match}', 'zip', f'{DST}/{match}')
    clips = sorted({key(f)[0] for f in pick})
    print(f'{match}: {len(imgs)} -> {len(pick)} frames ' +
          ', '.join(f'clip{c}: {sum(key(f)[0] == c for f in pick)}' for c in clips))

with open(f'{DST}/selection.csv', 'w', newline='') as fh:
    w = csv.DictWriter(fh, fieldnames=['match', 'file']); w.writeheader(); w.writerows(rows)
print(f'\nTotal: {len(rows)} frames in {DST}  (list: selection.csv)')